# Paper 4 — 04 · Gemma Scope SAE feature analysis (H1e)

**Gemma anchor only — corroboration.** Load pretrained Gemma Scope JumpReLU residual SAEs (no training), identify detection features (separate harm_en vs benign_en) and refusal features (separate refusal vs compliance), and compare firing on EN vs RO harmful prompts across the bands. H1e: detection features under-fire on RO in the detection band.

**Output:** `results/gemma-2-2b/sae_features.json`.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


In [ ]:
assert short == 'gemma-2-2b', 'H1e is Gemma-only (Gemma Scope SAEs).'

## Load Gemma Scope SAEs (per layer)

In [ ]:
from sae_lens import SAE
# repo = google/gemma-scope-2b-pt-res (or -it-res if coverage ok; see plan §13.1)
# load one SAE per band layer; hook = resid_post.


## Encode cell activations -> SAE latents

In [ ]:
from sae_utils import difference_in_means_features, en_ro_firing_gap
# det_feats = difference_in_means_features(harm_en_lat, benign_en_lat)
# ref_feats = difference_in_means_features(refusal_lat, comply_lat)


## EN vs RO firing comparison + width ablation

In [ ]:
# gap_det = en_ro_firing_gap(harm_en_lat, harm_ro_lat, det_feats)
# gap_ref = en_ro_firing_gap(harm_en_lat, harm_ro_lat, ref_feats)
# Repeat at a second SAE width; conclusion must survive (plan §8).
